
---

## 🧩 一、Level 1 问题类型分析（按能力分类）

根据 query + steps + answer 把这些任务分成了 **8 大类能力场景**，每类对应一个或多个工具（Tool）+ 智能体（Agent）。

| 能力类型                  | 示例 task                                          | 核心工具                                                                | 对应 Agent 角色                      |
| --------------------- | ------------------------------------------------ | ------------------------------------------------------------------- | -------------------------------- |
| **① 数学与逻辑推理类**        | 金条切割、巧克力切割、1+1                                   | `MathTool`（解析数学表达式、组合逻辑）                                            | `LogicAgent`（数理推理专家）             |
| **② 知识问答 / 常识类**      | Numpy 分布、F 展平、乔布斯与 iPhone                        | `PythonExecTool`（执行代码）+ `KnowledgeBaseTool`（百科检索）                   | `KnowledgeAgent`（常识与代码混合推理者）     |
| **③ 网页检索类**           | 京东金融板块、京东科技使命、零售云解决方案                            | `WebBrowserTool`（网页爬取+DOM解析）                                        | `WebSearchAgent`（网页阅读与结构化提取）     |
| **④ GitHub 结构解析类**    | Django 最低 Python 版本、Spark PR 字段、claude-code tags | `GitHubTool`（调用GitHub API或requests爬取）                               | `RepoAnalyzerAgent`（开源仓库信息提取者）   |
| **⑤ 文件内容解析类**         | 统计 log 文件、PDF 检索、图片数值计算、音频识别歌词                   | `FileSystemTool`（文件IO）+ `OCRTool` + `AudioRecognizer` + `PDFReader` | `FileAnalyzerAgent`（本地文件分析者）     |
| **⑥ 中文网页结构理解类（JD生态）** | 京东健康、京东产发、京东招聘等                                  | `JDNavigatorTool`（针对 JD 域名优化的爬虫）                                    | `JDInsightAgent`（专门负责JD相关站点结构理解） |
| **⑦ 知识图谱/实体消歧类**      | 苹果(Apple Inc.)、乔布斯、iPhone、iOS                    | `EntityLinkerTool`（基于上下文的实体选择）                                      | `SemanticAgent`（语义理解专家）          |
| **⑧ 计算/代码执行类**        | 生成矩阵、计算距离次数、保存文件后输出大小                            | `PythonRunner`（安全代码沙盒）                                              | `CodeExecutorAgent`（负责代码生成与运行）   |

---

## 🛠 二、工具（Tools）设计清单与功能说明

以下是每个工具的**实现目标与调用接口形式**（便于你直接开发 `tools/` 目录）：

### 1️⃣ MathTool

* **功能**：解析数学文字题（如“切金条最少几刀”）
* **方法**：

  ```python
  def solve_math_problem(text: str) -> str:
      # 可用 sympy / 自定义规则引擎解析
  ```

### 2️⃣ WebBrowserTool

* **功能**：浏览网页、定位元素、提取文本
* **接口**：

  ```python
  def fetch_page_content(url: str) -> str
  def extract_info_from_html(html: str, query: str) -> str
  ```
* **应用场景**：所有京东、京东金融、京东科技官网类题目。

### 3️⃣ GitHubTool

* **功能**：访问 GitHub 项目页面、获取 tags、PR 内容
* **接口**：

  ```python
  def get_repo_tags(repo: str) -> list[str]
  def get_pr_info(repo: str, pr_id: int) -> str
  ```
* **应用**：Django、Spark、Anthropic 相关题。

### 4️⃣ FileSystemTool

* **功能**：读取目录结构、统计文件数量
* **接口**：

  ```python
  def count_files_by_type(path: str, ext: str) -> int
  ```

### 5️⃣ OCRTool

* **功能**：从图片提取数值区域文本
* **接口**：

  ```python
  def extract_text_from_image(image_path: str) -> str
  ```

### 6️⃣ AudioRecognizer

* **功能**：识别 mp3 歌曲歌词内容（ASR + 歌曲匹配）
* **接口**：

  ```python
  def recognize_lyrics(audio_file: str) -> str
  ```

### 7️⃣ PDFReader

* **功能**：解析 PDF 文本内容
* **接口**：

  ```python
  def extract_text_from_pdf(path: str) -> str
  ```

### 8️⃣ PythonExecTool / PythonRunner

* **功能**：运行简单 numpy/pandas 代码块并输出结果
* **接口**：

  ```python
  def execute_code(code: str) -> str
  ```

### 9️⃣ KnowledgeBaseTool

* **功能**：调用百度百科、Wikipedia等API获取知识文本
* **接口**：

  ```python
  def query_knowledge_base(keyword: str) -> str
  ```

### 🔟 EntityLinkerTool

* **功能**：根据上下文和候选实体列表选出正确实体
* **接口**：

  ```python
  def disambiguate_entities(context: str, candidates: dict) -> str
  ```

---

## 🤖 三、Agent 设计（顶层逻辑）

Level 1 可设计成以下几个核心 Agent：

| Agent 名称              | 职责            | 调用工具                                                   | 工作流说明                  |
| --------------------- | ------------- | ------------------------------------------------------ | ---------------------- |
| **LogicAgent**        | 数学逻辑/常识题      | MathTool                                               | 输入→识别问题类型→推理输出         |
| **WebSearchAgent**    | 打开网页并定位内容     | WebBrowserTool                                         | 针对 query 含 URL 的题      |
| **JDInsightAgent**    | 专门处理京东生态相关问题  | JDNavigatorTool + WebBrowserTool                       | 识别“京东”“京东健康”“京东产发”等关键词 |
| **RepoAnalyzerAgent** | GitHub/开源项目相关 | GitHubTool                                             | 提取PR信息/tags/README     |
| **FileAnalyzerAgent** | 本地文件相关        | OCRTool + PDFReader + FileSystemTool + AudioRecognizer | 对本地文件路径调用合适子工具         |
| **CodeExecutorAgent** | 代码生成 + 执行验证   | PythonExecTool                                         | 适合矩阵还原类问题              |
| **SemanticAgent**     | 实体理解/语义推理     | EntityLinkerTool                                       | 用于文本中实体消歧问题            |
| **KnowledgeAgent**    | 常识问答/百科查询     | KnowledgeBaseTool                                      | 用于百科类题，如F1车手积分、石炭纪昆虫   |

---

## ⚙️ 四、Level 1 的总体推理流程（System Workflow）

```plaintext
用户 Query → LevelRouterAgent 判断类型 →
   ↳ 数学/逻辑类 → LogicAgent
   ↳ 含 URL → WebSearchAgent / JDInsightAgent
   ↳ 含 GitHub → RepoAnalyzerAgent
   ↳ 含 文件路径/本地文件 → FileAnalyzerAgent
   ↳ 含 “np.”、“Python” → CodeExecutorAgent
   ↳ 含 “候选实体” → SemanticAgent
   ↳ 其余 → KnowledgeAgent
→ 输出 answer
```

`LevelRouterAgent` 的判断逻辑核心是正则匹配 + 关键词判断，比如：

```python
if "jd.com" in query: return JDInsightAgent
elif "github" in query: return RepoAnalyzerAgent
elif "mp3" in query or ".jpg" in query or ".pdf" in query: return FileAnalyzerAgent
elif "矩阵" in query or "np." in query: return CodeExecutorAgent
elif any(k in query for k in ["候选实体", "上下文"]): return SemanticAgent
elif any(k in query for k in ["几刀", "切割", "1+1"]): return LogicAgent
else: return KnowledgeAgent
```

---

## 💡 五、优先开发建议（针对 Level 1）

优先完成这几个模块，就能解决 90% Level 1 的任务：

| 优先级 | 模块                                  |  对应任务数 | 开发难度 |
| :-: | :---------------------------------- | :----: | :--: |
|  🥇 | WebBrowserTool + JDInsightAgent     | 约 15 题 |   中  |
|  🥈 | MathTool + LogicAgent               |  约 4 题 |   低  |
|  🥉 | GitHubTool + RepoAnalyzerAgent      |  约 4 题 |   中  |
|  🏅 | FileAnalyzerAgent（整合 OCR+PDF+Audio） |  约 4 题 |   高  |
|  🧠 | KnowledgeBaseTool + KnowledgeAgent  |  约 5 题 |   中  |
|  ⚙️ | RouterAgent（分类器）                    | 所有任务入口 |   低  |

---

## 🚀 六、产出结构（推荐目录结构）

```
oxy_project/
├── agents/
│   ├── logic_agent.py
│   ├── web_search_agent.py
│   ├── jd_insight_agent.py
│   ├── repo_analyzer_agent.py
│   ├── file_analyzer_agent.py
│   ├── semantic_agent.py
│   ├── knowledge_agent.py
│   └── router_agent.py
├── tools/
│   ├── math_tool.py
│   ├── web_browser_tool.py
│   ├── github_tool.py
│   ├── filesystem_tool.py
│   ├── ocr_tool.py
│   ├── pdf_reader.py
│   ├── audio_recognizer.py
│   ├── python_exec_tool.py
│   ├── knowledge_base_tool.py
│   └── entity_linker_tool.py
└── main.py  # 调用router自动分配agent
```


